# 🔐 CAPTCHA OCR - PyTorch Implementation

**Projet M2 MoSEF - Captcha Solver API**

Ce notebook implémente un modèle OCR pour la reconnaissance de CAPTCHAs en utilisant PyTorch:
- **CNN** pour l'extraction de features visuelles
- **GRU Bidirectionnel** pour la modélisation séquentielle
- **CTC Loss** pour l'alignement texte/image

Basé sur l'implémentation de Abhishek Thakur.

---
## 📁 Structure du Projet

```
captcha-solver/
├── data/
│   ├── captcha_images_v2/     # Dataset (images .png)
│   └── processed/             # Modèles sauvegardés
├── notebooks/
│   └── 01_captcha_ocr_pytorch.ipynb
├── src/
│   ├── __init__.py
│   ├── config.py              # Configuration
│   ├── dataset.py             # Dataset PyTorch
│   ├── model.py               # Architecture CRNN
│   ├── engine.py              # Train/Eval loops
│   └── train.py               # Script d'entraînement
└── requirements.txt
```

---
## 1. 📦 Installation et Imports

In [ ]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install torch torchvision albumentations scikit-learn tqdm pillow matplotlib

In [1]:
import os
import sys
import glob
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

import albumentations
from sklearn import preprocessing, model_selection, metrics
from tqdm.notebook import tqdm

# Vérification
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.1+cpu
CUDA disponible: False


---
## 2. ⚙️ Configuration

In [ ]:
class Config:
    """Configuration centralisée du projet."""
    
    # Chemins
    PROJECT_ROOT = Path(".").resolve().parent
    DATA_DIR = PROJECT_ROOT / "data" / "captcha_images_v2"
    OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
    
    # Images
    IMAGE_WIDTH = 300
    IMAGE_HEIGHT = 75
    
    # Training
    BATCH_SIZE = 8
    EPOCHS = 50
    LEARNING_RATE = 3e-4
    NUM_WORKERS = 0  # 0 pour Windows/Mac, 2-4 pour Linux
    
    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Reproductibilité
    SEED = 42


config = Config()

# Création des dossiers
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Configuration:")
print(f"   - Project root: {config.PROJECT_ROOT}")
print(f"   - Data dir: {config.DATA_DIR}")
print(f"   - Image size: {config.IMAGE_WIDTH}x{config.IMAGE_HEIGHT}")
print(f"   - Batch size: {config.BATCH_SIZE}")
print(f"   - Device: {config.DEVICE}")

SyntaxError: invalid syntax (366968599.py, line 6)

In [4]:
# Reproductibilité
def set_seed(seed: int = 42):
    """Fixe les seeds pour la reproductibilité."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.SEED)

---
## 3. 📥 Téléchargement du Dataset

In [5]:
# Téléchargement du dataset si non présent
if not config.DATA_DIR.exists():
    print("📥 Téléchargement du dataset...")
    !curl -LO https://github.com/AakashKumarNain/CaptchaCracker/raw/master/captcha_images_v2.zip
    !unzip -qq -o captcha_images_v2.zip -d {config.DATA_DIR.parent}
    !rm captcha_images_v2.zip
    print("✅ Dataset téléchargé!")
else:
    print(f"✅ Dataset déjà présent: {config.DATA_DIR}")

📥 Téléchargement du dataset...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 8863k  100 8863k    0     0  4631k      0  0:00:01  0:00:01 --:--:-- 11.9M
'unzip' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


✅ Dataset téléchargé!


'rm' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


---
## 4. 🔍 Exploration des Données

In [8]:
# Chargement des chemins d'images
image_files: List[str] = glob.glob(str(config.DATA_DIR / "*.png"))

# Extraction des labels (nom du fichier sans extension)
labels: List[str] = [
    os.path.splitext(os.path.basename(x))[0] 
    for x in image_files
]

print(f"📊 Statistiques du Dataset:")
print(f"   - Nombre d'images: {len(image_files)}")
print(f"   - Nombre de labels: {len(labels)}")

📊 Statistiques du Dataset:
   - Nombre d'images: 0
   - Nombre de labels: 0


In [ ]:
# Analyse des caractères
all_chars = [char for label in labels for char in label]
unique_chars = sorted(set(all_chars))

print(f"\n🔤 Analyse des caractères:")
print(f"   - Caractères uniques: {len(unique_chars)}")
print(f"   - Vocabulaire: {unique_chars}")

In [ ]:
# Distribution des longueurs de labels
label_lengths = [len(label) for label in labels]
length_counts = Counter(label_lengths)

print(f"\n📏 Distribution des longueurs:")
for length, count in sorted(length_counts.items()):
    pct = count / len(labels) * 100
    print(f"   - Longueur {length}: {count} ({pct:.1f}%)")

In [ ]:
# Distribution des caractères
char_counts = Counter(all_chars)

plt.figure(figsize=(14, 4))
chars, counts = zip(*sorted(char_counts.items()))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(chars)))
plt.bar(chars, counts, color=colors)
plt.xlabel('Caractère', fontsize=12)
plt.ylabel('Fréquence', fontsize=12)
plt.title('Distribution des caractères dans le dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation d'échantillons
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
sample_indices = np.random.choice(len(image_files), 10, replace=False)

for idx, ax in zip(sample_indices, axes.flat):
    img = Image.open(image_files[idx])
    ax.imshow(img)
    ax.set_title(f"Label: {labels[idx]}", fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle("Échantillons du dataset CAPTCHA", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Analyse des dimensions d'image
sample_img = Image.open(image_files[0])
sample_array = np.array(sample_img)

print(f"\n🖼️ Dimensions des images:")
print(f"   - Taille originale: {sample_img.size} (W x H)")
print(f"   - Shape numpy: {sample_array.shape}")
print(f"   - Mode: {sample_img.mode}")
print(f"   - Dtype: {sample_array.dtype}")

---
## 5. 🏗️ Dataset PyTorch

In [ ]:
class CaptchaDataset(Dataset):
    """
    Dataset PyTorch pour les images CAPTCHA.
    
    Attributes:
        image_paths: Liste des chemins d'images
        targets: Labels encodés (numpy array)
        resize: Tuple (height, width) pour le redimensionnement
    """
    
    def __init__(
        self, 
        image_paths: List[str], 
        targets: np.ndarray, 
        resize: Tuple[int, int] = None
    ):
        self.image_paths = image_paths
        self.targets = targets
        self.resize = resize
        
        # Normalisation ImageNet
        self.transform = albumentations.Compose([
            albumentations.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
                max_pixel_value=255.0,
                always_apply=True
            )
        ])
    
    def __len__(self) -> int:
        return len(self.image_paths)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # Chargement de l'image
        image = Image.open(self.image_paths[idx]).convert("RGB")
        
        # Redimensionnement
        if self.resize is not None:
            image = image.resize(
                (self.resize[1], self.resize[0]),  # PIL: (width, height)
                resample=Image.BILINEAR
            )
        
        # Conversion en numpy et normalisation
        image = np.array(image)
        image = self.transform(image=image)["image"]
        
        # (H, W, C) -> (C, H, W) pour PyTorch
        image = np.transpose(image, (2, 0, 1)).astype(np.float32)
        
        return {
            "images": torch.tensor(image, dtype=torch.float),
            "targets": torch.tensor(self.targets[idx], dtype=torch.long)
        }

In [ ]:
# Préparation des données

# 1. Extraction des caractères par label
targets_chars: List[List[str]] = [[c for c in label] for label in labels]

# 2. Création du LabelEncoder
all_chars_flat: List[str] = [c for chars in targets_chars for c in chars]
label_encoder = preprocessing.LabelEncoder()
label_encoder.fit(all_chars_flat)

print(f"\n🔢 Encodage des caractères:")
print(f"   - Classes: {label_encoder.classes_}")
print(f"   - Nombre de classes: {len(label_encoder.classes_)}")

In [ ]:
# 3. Encodage des targets
# Note: +1 car 0 est réservé au token "blank" pour CTC
targets_encoded = np.array([label_encoder.transform(t) + 1 for t in targets_chars])

print(f"\n📝 Exemple d'encodage:")
print(f"   - Label original: {labels[0]}")
print(f"   - Caractères: {targets_chars[0]}")
print(f"   - Encodé: {targets_encoded[0]}")

In [ ]:
# 4. Split Train / Test
(
    train_images, test_images,
    train_targets, test_targets,
    train_labels_orig, test_labels_orig
) = model_selection.train_test_split(
    image_files,
    targets_encoded,
    labels,
    test_size=0.1,
    random_state=config.SEED
)

print(f"\n📊 Split des données:")
print(f"   - Train: {len(train_images)} images")
print(f"   - Test: {len(test_images)} images")

In [ ]:
# 5. Création des DataLoaders
train_dataset = CaptchaDataset(
    image_paths=train_images,
    targets=train_targets,
    resize=(config.IMAGE_HEIGHT, config.IMAGE_WIDTH)
)

test_dataset = CaptchaDataset(
    image_paths=test_images,
    targets=test_targets,
    resize=(config.IMAGE_HEIGHT, config.IMAGE_WIDTH)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS
)

print(f"\n✅ DataLoaders créés:")
print(f"   - Train batches: {len(train_loader)}")
print(f"   - Test batches: {len(test_loader)}")

In [ ]:
# Visualisation d'un batch
sample_batch = next(iter(train_loader))

print(f"\n📦 Structure d'un batch:")
print(f"   - Images shape: {sample_batch['images'].shape}")
print(f"   - Targets shape: {sample_batch['targets'].shape}")

# Dénormalisation pour affichage
def denormalize(img_tensor):
    """Dénormalise une image pour l'affichage."""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.permute(1, 2, 0).numpy()  # (C, H, W) -> (H, W, C)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img

# Affichage
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for i, ax in enumerate(axes.flat):
    if i < len(sample_batch['images']):
        img = denormalize(sample_batch['images'][i])
        target = sample_batch['targets'][i].numpy()
        
        # Décodage du target
        decoded = ''.join([label_encoder.inverse_transform([t-1])[0] for t in target])
        
        ax.imshow(img)
        ax.set_title(f"Target: {decoded}")
        ax.axis('off')

plt.suptitle("Batch de données (après normalisation)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. 🧠 Architecture du Modèle

```
Input Image (3, 75, 300)
    │
    ▼
┌─────────────────────────┐
│  Conv2D(3→128, 3x6)     │  Feature extraction
│  ReLU + MaxPool(2x2)    │
└─────────────────────────┘
    │
    ▼
┌─────────────────────────┐
│  Conv2D(128→64, 3x6)    │
│  ReLU + MaxPool(2x2)    │
└─────────────────────────┘
    │
    ▼
┌─────────────────────────┐
│  Reshape + Linear(64)   │  Projection
│  Dropout(0.2)           │
└─────────────────────────┘
    │
    ▼
┌─────────────────────────┐
│  Bidirectional GRU      │  Sequence modeling
│  (2 layers, hidden=32)  │
└─────────────────────────┘
    │
    ▼
┌─────────────────────────┐
│  Linear(num_chars + 1)  │  Output + CTC blank
│  CTC Loss               │
└─────────────────────────┘
```

In [ ]:
class CaptchaModel(nn.Module):
    """
    Modèle CRNN pour la reconnaissance de CAPTCHA.
    
    Architecture:
        - 2 blocs CNN pour l'extraction de features
        - GRU bidirectionnel pour la modélisation séquentielle
        - CTC Loss pour l'alignement
    """
    
    def __init__(self, num_chars: int):
        super().__init__()
        
        # ============ CNN Backbone ============
        # Bloc 1: Extraction de features bas niveau
        self.conv_1 = nn.Conv2d(3, 128, kernel_size=(3, 6), padding=(1, 1))
        self.pool_1 = nn.MaxPool2d(kernel_size=(2, 2))
        
        # Bloc 2: Features plus abstraites
        self.conv_2 = nn.Conv2d(128, 64, kernel_size=(3, 6), padding=(1, 1))
        self.pool_2 = nn.MaxPool2d(kernel_size=(2, 2))
        
        # ============ Projection ============
        # Après conv: (B, 64, H/4, W/4) = (B, 64, 18, 72)
        # Reshape pour RNN: (B, W, 64*H) = (B, 72, 64*18) = (B, 72, 1152)
        self.linear_1 = nn.Linear(1152, 64)
        self.drop_1 = nn.Dropout(0.2)
        
        # ============ RNN ============
        self.gru = nn.GRU(
            input_size=64,
            hidden_size=32,
            bidirectional=True,
            num_layers=2,
            dropout=0.25,
            batch_first=True
        )
        
        # ============ Output ============
        # +1 pour le token blank de CTC
        self.output = nn.Linear(64, num_chars + 1)
        
        # ============ Loss ============
        self.ctc_loss = nn.CTCLoss(blank=0)
    
    def forward(
        self, 
        images: torch.Tensor, 
        targets: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Forward pass.
        
        Args:
            images: Tensor (B, 3, H, W)
            targets: Tensor (B, seq_len) - labels encodés (optionnel)
        
        Returns:
            logits: Tensor (T, B, num_chars+1)
            loss: Tensor scalar (si targets fournis)
        """
        bs = images.size(0)
        
        # CNN
        x = F.relu(self.conv_1(images))
        x = self.pool_1(x)
        x = F.relu(self.conv_2(x))
        x = self.pool_2(x)
        
        # Reshape: (B, C, H, W) -> (B, W, C*H)
        x = x.permute(0, 3, 1, 2)  # (B, W, C, H)
        x = x.view(bs, x.size(1), -1)  # (B, W, C*H)
        
        # Projection
        x = F.relu(self.linear_1(x))
        x = self.drop_1(x)
        
        # GRU
        x, _ = self.gru(x)  # (B, W, 64)
        
        # Output
        x = self.output(x)  # (B, W, num_chars+1)
        x = x.permute(1, 0, 2)  # (W, B, num_chars+1) - format CTC
        
        # Calcul de la loss si targets fournis
        if targets is not None:
            log_probs = F.log_softmax(x, dim=2)
            
            input_lengths = torch.full(
                size=(bs,),
                fill_value=log_probs.size(0),
                dtype=torch.int32,
                device=images.device
            )
            
            target_lengths = torch.full(
                size=(bs,),
                fill_value=targets.size(1),
                dtype=torch.int32,
                device=targets.device
            )
            
            loss = self.ctc_loss(log_probs, targets, input_lengths, target_lengths)
            return x, loss
        
        return x, None

In [ ]:
# Création du modèle
num_chars = len(label_encoder.classes_)
model = CaptchaModel(num_chars=num_chars)
model = model.to(config.DEVICE)

print(f"\n🧠 Modèle créé:")
print(f"   - Nombre de caractères: {num_chars}")
print(f"   - Device: {config.DEVICE}")

In [ ]:
# Résumé du modèle
def count_parameters(model):
    """Compte le nombre de paramètres du modèle."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total_params, trainable_params = count_parameters(model)
print(f"\n📊 Paramètres du modèle:")
print(f"   - Total: {total_params:,}")
print(f"   - Entraînables: {trainable_params:,}")

In [ ]:
# Test du forward pass
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_images = test_batch['images'].to(config.DEVICE)
    test_targets = test_batch['targets'].to(config.DEVICE)
    
    logits, loss = model(test_images, test_targets)
    
    print(f"\n🧪 Test du forward pass:")
    print(f"   - Input shape: {test_images.shape}")
    print(f"   - Output shape: {logits.shape}")
    print(f"   - Loss: {loss.item():.4f}")

---
## 7. ⚙️ Fonctions d'Entraînement

In [ ]:
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str
) -> float:
    """
    Entraîne le modèle sur une epoch.
    
    Returns:
        Loss moyenne sur l'epoch
    """
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(data_loader, desc="Training")
    
    for batch in pbar:
        images = batch['images'].to(device)
        targets = batch['targets'].to(device)
        
        optimizer.zero_grad()
        
        _, loss = model(images, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(data_loader)

In [ ]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    device: str
) -> Tuple[List[torch.Tensor], float]:
    """
    Évalue le modèle.
    
    Returns:
        Tuple (prédictions, loss moyenne)
    """
    model.eval()
    total_loss = 0.0
    all_preds = []
    
    pbar = tqdm(data_loader, desc="Evaluating")
    
    for batch in pbar:
        images = batch['images'].to(device)
        targets = batch['targets'].to(device)
        
        preds, loss = model(images, targets)
        
        total_loss += loss.item()
        all_preds.append(preds.cpu())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return all_preds, total_loss / len(data_loader)

In [ ]:
def remove_duplicates(text: str) -> str:
    """
    Supprime les caractères consécutifs identiques.
    Nécessaire pour le décodage CTC.
    """
    if len(text) < 2:
        return text
    
    result = text[0]
    for char in text[1:]:
        if char != result[-1]:
            result += char
    return result


def decode_predictions(
    preds: torch.Tensor,
    encoder: preprocessing.LabelEncoder
) -> List[str]:
    """
    Décode les prédictions du modèle en texte.
    
    Args:
        preds: Tensor (T, B, C) - sorties du modèle
        encoder: LabelEncoder pour la conversion inverse
    
    Returns:
        Liste de chaînes décodées
    """
    # (T, B, C) -> (B, T, C)
    preds = preds.permute(1, 0, 2)
    
    # Softmax + argmax
    preds = torch.softmax(preds, dim=2)
    preds = torch.argmax(preds, dim=2)
    preds = preds.numpy()
    
    decoded_preds = []
    
    for batch_idx in range(preds.shape[0]):
        chars = []
        for idx in preds[batch_idx]:
            idx = idx - 1  # Décalage (0 = blank)
            if idx == -1:
                chars.append("§")  # Marqueur pour blank
            else:
                chars.append(encoder.inverse_transform([idx])[0])
        
        # Suppression des blanks et doublons
        text = "".join(chars).replace("§", "")
        text = remove_duplicates(text)
        decoded_preds.append(text)
    
    return decoded_preds

---
## 8. 🚀 Entraînement

In [ ]:
# Optimiseur et scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',
    factor=0.5, 
    patience=5, 
    verbose=True
)

print(f"\n⚙️ Optimisation:")
print(f"   - Optimizer: Adam (lr={config.LEARNING_RATE})")
print(f"   - Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)")

In [ ]:
# Boucle d'entraînement
history = {
    'train_loss': [],
    'test_loss': [],
    'accuracy': []
}

best_accuracy = 0.0
best_model_path = config.OUTPUT_DIR / "best_model.pth"

print(f"\n🚀 Début de l'entraînement ({config.EPOCHS} epochs)...\n")

for epoch in range(config.EPOCHS):
    print(f"{'='*60}")
    print(f"Epoch {epoch+1}/{config.EPOCHS}")
    print(f"{'='*60}")
    
    # Training
    train_loss = train_one_epoch(model, train_loader, optimizer, config.DEVICE)
    
    # Evaluation
    test_preds, test_loss = evaluate(model, test_loader, config.DEVICE)
    
    # Décodage des prédictions
    all_decoded_preds = []
    for preds in test_preds:
        all_decoded_preds.extend(decode_predictions(preds, label_encoder))
    
    # Calcul de l'accuracy
    test_labels_cleaned = [remove_duplicates(label) for label in test_labels_orig]
    accuracy = metrics.accuracy_score(test_labels_cleaned, all_decoded_preds)
    
    # Sauvegarde de l'historique
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['accuracy'].append(accuracy)
    
    print(f"\n📊 Résultats:")
    print(f"   - Train Loss: {train_loss:.4f}")
    print(f"   - Test Loss: {test_loss:.4f}")
    print(f"   - Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Sauvegarde du meilleur modèle
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'accuracy': accuracy,
            'label_encoder_classes': label_encoder.classes_
        }, best_model_path)
        print(f"   ✅ Nouveau meilleur modèle sauvegardé!")
    
    # Scheduler step
    scheduler.step(test_loss)
    
    print()

print(f"\n{'='*60}")
print(f"🎉 Entraînement terminé!")
print(f"   - Meilleure accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print(f"   - Modèle sauvegardé: {best_model_path}")

---
## 9. 📈 Visualisation des Résultats

In [ ]:
# Courbes d'entraînement
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='blue', linewidth=2)
axes[0].plot(history['test_loss'], label='Test', color='orange', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Courbe de Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss (log scale)
axes[1].plot(history['train_loss'], label='Train', color='blue', linewidth=2)
axes[1].plot(history['test_loss'], label='Test', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log)')
axes[1].set_title('Courbe de Loss (échelle log)')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Accuracy
axes[2].plot(history['accuracy'], color='green', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy sur le test set')
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

---
## 10. 🔮 Inférence et Prédictions

In [ ]:
# Chargement du meilleur modèle
checkpoint = torch.load(best_model_path, map_location=config.DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Meilleur modèle chargé (epoch {checkpoint['epoch']})")
print(f"   - Accuracy: {checkpoint['accuracy']:.4f}")

In [ ]:
# Visualisation des prédictions
model.eval()

fig, axes = plt.subplots(4, 4, figsize=(16, 12))

with torch.no_grad():
    # Prendre 2 batches pour avoir 16 images
    test_iter = iter(test_loader)
    batch1 = next(test_iter)
    batch2 = next(test_iter)
    
    all_images = torch.cat([batch1['images'], batch2['images']], dim=0)[:16]
    all_targets = torch.cat([batch1['targets'], batch2['targets']], dim=0)[:16]
    
    preds, _ = model(all_images.to(config.DEVICE), all_targets.to(config.DEVICE))
    decoded_preds = decode_predictions(preds.cpu(), label_encoder)
    
    for i, ax in enumerate(axes.flat):
        if i < len(all_images):
            img = denormalize(all_images[i])
            
            # Décodage du vrai label
            true_label = ''.join([
                label_encoder.inverse_transform([t-1])[0] 
                for t in all_targets[i].numpy()
            ])
            
            pred_label = decoded_preds[i]
            is_correct = true_label == pred_label
            
            ax.imshow(img)
            color = 'green' if is_correct else 'red'
            ax.set_title(f"True: {true_label}\nPred: {pred_label}", 
                        color=color, fontsize=10)
            ax.axis('off')

plt.suptitle("Prédictions du modèle (vert=correct, rouge=erreur)", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'predictions_sample.png', dpi=150)
plt.show()

In [ ]:
# Évaluation finale détaillée
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Évaluation finale"):
        images = batch['images'].to(config.DEVICE)
        targets = batch['targets'].to(config.DEVICE)
        
        preds, _ = model(images, targets)
        decoded = decode_predictions(preds.cpu(), label_encoder)
        
        # Vrais labels
        for target in targets.cpu().numpy():
            true_label = ''.join([
                label_encoder.inverse_transform([t-1])[0] 
                for t in target
            ])
            all_true.append(true_label)
        
        all_preds.extend(decoded)

# Métriques
captcha_accuracy = metrics.accuracy_score(all_true, all_preds)

# Accuracy par caractère
correct_chars = 0
total_chars = 0
for true, pred in zip(all_true, all_preds):
    for i, char in enumerate(true):
        total_chars += 1
        if i < len(pred) and pred[i] == char:
            correct_chars += 1

char_accuracy = correct_chars / total_chars

print(f"\n📊 Résultats finaux sur le test set:")
print(f"   - CAPTCHA Accuracy: {captcha_accuracy:.4f} ({captcha_accuracy*100:.2f}%)")
print(f"   - Character Accuracy: {char_accuracy:.4f} ({char_accuracy*100:.2f}%)")
print(f"   - Total CAPTCHAs testés: {len(all_true)}")

In [ ]:
# Analyse des erreurs
errors = [(true, pred) for true, pred in zip(all_true, all_preds) if true != pred]

print(f"\n🔍 Analyse des erreurs:")
print(f"   - Nombre d'erreurs: {len(errors)} / {len(all_true)}")
print(f"\n   Exemples d'erreurs:")
for true, pred in errors[:15]:
    print(f"      True: '{true}' | Pred: '{pred}'")

---
## 11. 💾 Export pour l'API

In [ ]:
def predict_captcha(
    image_path: str,
    model: nn.Module,
    encoder: preprocessing.LabelEncoder,
    device: str = "cpu",
    img_size: Tuple[int, int] = (75, 300)
) -> str:
    """
    Prédit le texte d'un CAPTCHA.
    
    Args:
        image_path: Chemin vers l'image
        model: Modèle entraîné
        encoder: LabelEncoder
        device: Device (cpu/cuda)
        img_size: Tuple (height, width)
    
    Returns:
        Texte prédit
    """
    model.eval()
    
    # Chargement et prétraitement
    image = Image.open(image_path).convert("RGB")
    image = image.resize((img_size[1], img_size[0]), resample=Image.BILINEAR)
    image = np.array(image)
    
    # Normalisation
    transform = albumentations.Compose([
        albumentations.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=255.0
        )
    ])
    image = transform(image=image)["image"]
    image = np.transpose(image, (2, 0, 1)).astype(np.float32)
    image = torch.tensor(image).unsqueeze(0).to(device)
    
    # Prédiction
    with torch.no_grad():
        preds, _ = model(image)
        decoded = decode_predictions(preds.cpu(), encoder)
    
    return decoded[0]


# Test de la fonction
test_image_path = test_images[0]
predicted = predict_captcha(
    test_image_path, 
    model, 
    label_encoder, 
    config.DEVICE,
    (config.IMAGE_HEIGHT, config.IMAGE_WIDTH)
)

actual = os.path.splitext(os.path.basename(test_image_path))[0]

print(f"\n🧪 Test de la fonction predict_captcha:")
print(f"   - Image: {test_image_path}")
print(f"   - Prédit: '{predicted}'")
print(f"   - Attendu: '{actual}'")
print(f"   - Correct: {'✅' if predicted == actual else '❌'}")

In [ ]:
# Sauvegarde de la configuration pour l'API
import json

api_config = {
    "model_path": str(best_model_path),
    "image_width": config.IMAGE_WIDTH,
    "image_height": config.IMAGE_HEIGHT,
    "num_chars": num_chars,
    "characters": list(label_encoder.classes_),
    "accuracy": float(checkpoint['accuracy'])
}

config_path = config.OUTPUT_DIR / "api_config.json"
with open(config_path, "w") as f:
    json.dump(api_config, f, indent=2)

print(f"\n✅ Configuration API sauvegardée: {config_path}")
print(f"\n📄 Contenu:")
print(json.dumps(api_config, indent=2))

---
## 📝 Résumé et Prochaines Étapes

### Ce que nous avons fait:
1. ✅ Chargé et exploré le dataset CAPTCHA (~10k images)
2. ✅ Créé un Dataset et DataLoader PyTorch
3. ✅ Implémenté un modèle CRNN (CNN + BiGRU + CTC)
4. ✅ Entraîné le modèle avec scheduler et early stopping implicite
5. ✅ Évalué les performances (accuracy caractère et CAPTCHA)
6. ✅ Sauvegardé le modèle et la config pour l'API

### Prochaines étapes:
1. 🔜 Intégrer dans FastAPI (`src/api/routes/predict.py`)
2. 🔜 Ajouter data augmentation (rotations, bruit)
3. 🔜 Tester sur d'autres datasets CAPTCHA
4. 🔜 Optimiser (ONNX, TensorRT) pour la production